# Chapter 17: Comparing Groups and A/B Tests

The programs from Chapter 17 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## The Experiment

`ab_summary.py`

In [1]:
import pandas as pd

sessions = pd.read_csv("../data/sessions.csv")

summary = sessions.groupby("variant").agg(
    sessions=("session", "count"),
    conversions=("converted", "sum"),
    revenue=("revenue", "sum"),
)
summary["rate"] = summary["conversions"] / summary["sessions"]
summary["revenue_per_session"] = summary["revenue"] / summary["sessions"]
print(summary.round(4))
lift = summary.loc["B", "rate"] / summary.loc["A", "rate"] - 1
print(f"B converts {lift:.1%} better than A")

         sessions  conversions   revenue    rate  revenue_per_session
variant                                                              
A           10024          402  27372.98  0.0401               2.7307
B            9976          476  31845.27  0.0477               3.1922
B converts 19.0% better than A


## The Bootstrap: How Precise Is the Difference?

`bootstrap.py`

In [2]:
import numpy as np
import pandas as pd

sessions = pd.read_csv("../data/sessions.csv")
a = sessions.loc[sessions["variant"] == "A", "converted"].to_numpy()
b = sessions.loc[sessions["variant"] == "B", "converted"].to_numpy()

rng = np.random.default_rng(1)
diffs = np.empty(5_000)
for i in range(len(diffs)):
    diffs[i] = rng.choice(b, len(b)).mean() - rng.choice(a, len(a)).mean()

low, high = np.percentile(diffs, [2.5, 97.5])
print(f"difference in conversion rate: {b.mean() - a.mean():.4f}")
print(f"95% bootstrap interval: {low:.4f} to {high:.4f}")
print("interval excludes zero:", bool(low > 0 or high < 0))

difference in conversion rate: 0.0076
95% bootstrap interval: 0.0020 to 0.0132
interval excludes zero: True


## The Permutation Test: Could It Be Luck?

`permutation.py`

In [3]:
import numpy as np
import pandas as pd

sessions = pd.read_csv("../data/sessions.csv")
converted = sessions["converted"].to_numpy()
is_b = (sessions["variant"] == "B").to_numpy()
observed = converted[is_b].mean() - converted[~is_b].mean()

rng = np.random.default_rng(2)
count = 0
trials = 5_000
for _ in range(trials):
    shuffled = rng.permutation(is_b)  # pretend the labels mean nothing
    diff = converted[shuffled].mean() - converted[~shuffled].mean()
    count += abs(diff) >= abs(observed)

print(f"observed difference: {observed:.4f}")
print(f"p-value: {count / trials:.3f}")

observed difference: 0.0076
p-value: 0.010
